# DENTRAT — Comprehensive Performance Evaluation

Evaluate your trained dental anomaly detection model (Faster R-CNN ResNet50-FPN) with full metrics, visualizations, and an HTML report.

---

## Instructions

1. **Upload your trained model file** (`.pth`) when prompted in Section 2
2. **Upload your test dataset annotations** (COCO JSON or CSV like Dataset_Batch2)
3. **Upload test images** (and optionally a few sample images for visualization)
4. **Click Run All** (Runtime → Run all) to generate the complete performance report

**Output:** All figures saved to `metrics_output/` + `metrics_output/performance_report.html`

---

**Architecture:** Faster R-CNN ResNet50-FPN | **Classes:** 7 anomaly types (IDs 1–7) | **Input:** 416×416


## 1. Install Dependencies


In [ ]:
!pip install -q torch torchvision albumentations opencv-python-headless scikit-learn matplotlib seaborn pandas tqdm pillow tabulate

import os, json, glob, re, time, copy, warnings, zipfile, io, base64
from collections import defaultdict
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from sklearn.metrics import (
    precision_recall_fscore_support, confusion_matrix, roc_auc_score,
    matthews_corrcoef, balanced_accuracy_score, accuracy_score,
    precision_recall_curve, roc_curve, auc,
)

warnings.filterwarnings("ignore")
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
sns.set_palette("husl")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Upload Files

Upload when prompted:
- **Model checkpoint** (`.pth`)
- **Test annotations** — `_annotations.csv` or COCO JSON (e.g. `_annotations.coco.json`)
- **Test images** — a `.zip` containing a `test/` folder with images, or individual image files

**Zip tip:** If your dataset looks like `Dataset_Batch2/test/*.jpg` + `test/_annotations.csv`, upload the CSV separately and zip the `test/` folder (or entire dataset). The notebook will find images inside nested folders automatically.

- **Sample images** (optional) — for qualitative error analysis


In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — set paths manually below.")

WORK_DIR = "/content/dental_metrics_eval"
METRICS_DIR = os.path.join(WORK_DIR, "metrics_output")
TEST_IMAGES_DIR = os.path.join(WORK_DIR, "test_images")
SAMPLE_IMAGES_DIR = os.path.join(WORK_DIR, "sample_images")
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")


def extract_zip_recursive(zip_path, dest_dir):
    """Extract zip; if it contains nested zips, extract those too."""
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    for root, _, files_in_dir in os.walk(dest_dir):
        for f in files_in_dir:
            if f.lower().endswith(".zip"):
                nested = os.path.join(root, f)
                extract_zip_recursive(nested, root)
                os.remove(nested)


def scan_image_dirs(base_dir):
    """List all directories under base_dir that contain images (recursive)."""
    found = []
    if not os.path.isdir(base_dir):
        return found
    for root, _, files_in_dir in os.walk(base_dir):
        n = sum(1 for f in files_in_dir if f.lower().endswith(IMAGE_EXTS))
        if n:
            found.append((root, n))
    return sorted(found, key=lambda x: -x[1])


for d in [WORK_DIR, METRICS_DIR, TEST_IMAGES_DIR, SAMPLE_IMAGES_DIR]:
    os.makedirs(d, exist_ok=True)
os.chdir(WORK_DIR)

if IN_COLAB:
    print("Upload your .pth model file:")
    model_upload = files.upload()
    MODEL_PATH = os.path.join(WORK_DIR, list(model_upload.keys())[0])
    print(f"Model saved: {MODEL_PATH}")

    print("\nUpload test annotations (CSV or COCO JSON):")
    ann_upload = files.upload()
    ANN_PATH = os.path.join(WORK_DIR, list(ann_upload.keys())[0])
    print(f"Annotations saved: {ANN_PATH}")

    print("\nUpload test images (zip with test/ folder, or individual images):")
    img_upload = files.upload()
    for fname, data in img_upload.items():
        dest = os.path.join(TEST_IMAGES_DIR, fname)
        with open(dest, "wb") as f:
            f.write(data)
        if fname.lower().endswith(".zip"):
            extract_zip_recursive(dest, TEST_IMAGES_DIR)
    img_dirs = scan_image_dirs(TEST_IMAGES_DIR)
    print(f"Uploaded {len(img_upload)} file(s). Image folders found:")
    for dpath, count in img_dirs[:10]:
        print(f"  {dpath} ({count} images)")
    if not img_dirs:
        print("  ⚠ No images found yet — check your zip structure.")

    print("\nUpload sample images for visualization (optional — Cancel to skip):")
    try:
        sample_upload = files.upload()
        for fname, data in sample_upload.items():
            with open(os.path.join(SAMPLE_IMAGES_DIR, fname), "wb") as f:
                f.write(data)
        if sample_upload:
            print(f"Uploaded {len(sample_upload)} sample image(s)")
    except Exception:
        print("No sample images uploaded.")
else:
    MODEL_PATH = "dental_model_v3.pth"
    ANN_PATH = "test/_annotations.csv"
    print("Set MODEL_PATH and ANN_PATH manually for local runs.")


## 3. Configuration


In [ ]:
CLASS_NAMES = {
    1: "Caries",
    2: "Impacted Teeth",
    3: "Broken Down Crown/Root",
    4: "Infection",
    5: "Fractured Teeth",
    6: "Periodontal Bone Loss",
    7: "Other Abnormalities",
}
CLASS_IDS = list(CLASS_NAMES.keys())
CLASS_LABELS = [CLASS_NAMES[i] for i in CLASS_IDS]

NUM_CLASSES = 8
IMAGE_SIZE = 416
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
IOU_THRESHOLDS_MAP = np.arange(0.5, 1.0, 0.05)

LABEL_MAP = {
    0: None, 1: 1, 2: 2, 3: 4, 4: 5, 5: 3, 6: 6, 7: 7, 8: None,
}

KEYWORD_MAP = {
    1: ["caries", "cavity", "cavities", "decay"],
    2: ["impacted", "impaction"],
    3: ["broken crown", "broken down", "crown", "root"],
    4: ["infection", "abscess", "infected"],
    5: ["fractured", "fracture", "crack"],
    6: ["periodontal", "bone loss", "periodontitis"],
    7: ["other", "abnormal", "anomaly", "misc"],
}
SKIP_KEYWORDS = ["healthy", "normal", "no finding", "background", "none"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 4. Dataset Loading (COCO JSON & CSV Auto-Detect)


In [ ]:
def parse_class_id(raw_class):
    if raw_class is None or (isinstance(raw_class, float) and np.isnan(raw_class)):
        return -1
    if isinstance(raw_class, str):
        raw_class = raw_class.strip()
        if not raw_class:
            return -1
        try:
            return int(float(raw_class))
        except ValueError:
            pass
        name_to_id = {v.lower(): k for k, v in CLASS_NAMES.items()}
        if raw_class.lower() in name_to_id:
            return name_to_id[raw_class.lower()]
        for target_id, keywords in KEYWORD_MAP.items():
            low = raw_class.lower()
            if any(kw in low or low in kw for kw in keywords):
                return target_id
        return -1
    try:
        return int(raw_class)
    except (TypeError, ValueError):
        return -1


def map_to_model_class(raw_value):
    """Map CSV/COCO class value to model class 1-7 (or None to skip)."""
    src_id = parse_class_id(raw_value)
    if src_id in LABEL_MAP:
        return LABEL_MAP[src_id]
    if src_id in CLASS_NAMES:
        return src_id
    # Roboflow / COCO often use 0-indexed category IDs
    if 0 <= src_id <= 6:
        return src_id + 1
    return None


def auto_map_category(cat_id, cat_name):
    name_lower = cat_name.lower().strip()
    for skip in SKIP_KEYWORDS:
        if skip in name_lower:
            return None
    for target_id, keywords in KEYWORD_MAP.items():
        for kw in keywords:
            if kw in name_lower or name_lower in kw:
                return target_id
    if cat_id in LABEL_MAP:
        return LABEL_MAP[cat_id]
    if cat_id in CLASS_NAMES:
        return cat_id
    if 0 <= cat_id <= 6:
        return cat_id + 1
    return None


def coco_bbox_to_pascal(bbox, img_w, img_h):
    x, y, w, h = bbox
    if max(x, y, w, h) <= 1.0:
        x, y, w, h = x * img_w, y * img_h, w * img_w, h * img_h
    return [x, y, x + w, y + h]


def detect_format(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return "csv"
    if ext == ".json":
        return "coco"
    with open(path, "r", encoding="utf-8") as f:
        head = f.read(256)
    if head.strip().startswith("{"):
        return "coco"
    return "csv"


def build_image_index(search_roots):
    """Recursively index all images by basename and normalized relative path."""
    index = {}
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, files_in_dir in os.walk(root):
            for fname in files_in_dir:
                if not fname.lower().endswith(IMAGE_EXTS):
                    continue
                full = os.path.join(dirpath, fname)
                rel = os.path.relpath(full, root).replace("\\", "/")
                index.setdefault(fname, []).append(full)
                index.setdefault(rel, []).append(full)
                index.setdefault(os.path.basename(rel), []).append(full)
    return index


def find_best_images_dir(search_roots):
    """Pick the directory with the most images (prefers folders named 'test')."""
    best_dir, best_count = TEST_IMAGES_DIR, 0
    candidates = []
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, files_in_dir in os.walk(root):
            n = sum(1 for f in files_in_dir if f.lower().endswith(IMAGE_EXTS))
            if n:
                score = n + (10 if os.path.basename(dirpath).lower() == "test" else 0)
                candidates.append((dirpath, n, score))
    if candidates:
        best_dir = max(candidates, key=lambda x: x[2])[0]
    return best_dir


def resolve_image_path(filename, image_index, images_dir):
    """Find image file from CSV filename — handles test/foo.jpg, subfolders, etc."""
    filename = str(filename).strip().replace("\\", "/")
    basename = os.path.basename(filename)
    candidates = []

    def add(p):
        if p and os.path.isfile(p) and p not in candidates:
            candidates.append(p)

    for key in [filename, basename, filename.lstrip("./")]:
        for p in image_index.get(key, []):
            add(p)
    add(os.path.join(images_dir, filename))
    add(os.path.join(images_dir, basename))
    parts = filename.split("/")
    if len(parts) > 1:
        add(os.path.join(images_dir, *parts))
        add(os.path.join(images_dir, parts[-1]))
    if "test/" in filename:
        add(os.path.join(images_dir, filename.split("test/", 1)[-1]))

    return candidates[0] if candidates else None


def get_row_value(row, *keys, default=None):
    for k in keys:
        if k in row.index and pd.notna(row[k]):
            return row[k]
    return default


def parse_csv_bbox(row):
    """Parse bbox from CSV — supports Pascal VOC, COCO w/h, and normalized coords."""
    img_w = float(get_row_value(row, "width", "img_width", "image_width", default=0) or 0)
    img_h = float(get_row_value(row, "height", "img_height", "image_height", default=0) or 0)

    xmin = get_row_value(row, "xmin", "x_min", "x1", "left")
    ymin = get_row_value(row, "ymin", "y_min", "y1", "top")
    xmax = get_row_value(row, "xmax", "x_max", "x2", "right")
    ymax = get_row_value(row, "ymax", "y_max", "y2", "bottom")
    width = get_row_value(row, "width_box", "box_width", "w")
    height = get_row_value(row, "height_box", "box_height", "h")

    if xmin is None or ymin is None:
        return None

    xmin, ymin = float(xmin), float(ymin)

    if xmax is not None and ymax is not None:
        xmax, ymax = float(xmax), float(ymax)
    elif width is not None and height is not None:
        xmax = xmin + float(width)
        ymax = ymin + float(height)
    else:
        return None

    # Normalized 0-1 coordinates (common in Roboflow exports)
    if img_w > 1 and img_h > 1 and max(xmin, ymin, xmax, ymax) <= 1.0:
        xmin, ymin, xmax, ymax = xmin * img_w, ymin * img_h, xmax * img_w, ymax * img_h
    elif img_w > 1 and img_h > 1 and max(xmin, ymin, xmax, ymax) <= 100.0 and max(xmin, ymin, xmax, ymax) <= min(img_w, img_h):
        # Percentage coords (0-100)
        if xmax <= 100 and ymax <= 100:
            xmin, ymin, xmax, ymax = xmin/100*img_w, ymin/100*img_h, xmax/100*img_w, ymax/100*img_h

    if xmax <= xmin or ymax <= ymin:
        return None
    return [xmin, ymin, xmax, ymax]


def load_csv_annotations(csv_path, images_dir, image_index):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    print(f"CSV columns: {list(df.columns)}")
    print(f"CSV rows: {len(df)}")
    if len(df):
        print(f"Sample row: {df.iloc[0].to_dict()}")

    grouped = defaultdict(list)
    skipped_class = 0
    skipped_bbox = 0
    unmapped_classes = set()

    for _, row in df.iterrows():
        filename = get_row_value(row, "filename", "file_name", "filepath", "file_path", "image", "image_path", "img")
        if filename is None:
            continue
        filename = str(filename).strip()

        raw_cls = get_row_value(row, "class", "class_id", "category", "category_id", "label", "name", "type")
        mapped = map_to_model_class(raw_cls)
        if mapped is None or mapped not in CLASS_NAMES:
            skipped_class += 1
            unmapped_classes.add(str(raw_cls))
            continue

        bbox = parse_csv_bbox(row)
        if bbox is None:
            skipped_bbox += 1
            continue
        grouped[filename].append({"bbox": bbox, "label": mapped})

    samples = []
    missing_images = []
    for filename, anns in grouped.items():
        img_path = resolve_image_path(filename, image_index, images_dir)
        if img_path and len(anns) > 0:
            samples.append({"filename": filename, "path": img_path, "annotations": anns})
        else:
            missing_images.append(filename)

    print(f"CSV: loaded {len(samples)} images | skipped {skipped_class} rows (class) | skipped {skipped_bbox} rows (bbox)")
    if unmapped_classes:
        print(f"  Unmapped class values: {sorted(unmapped_classes)[:15]}")
    if missing_images:
        print(f"  ⚠ {len(missing_images)} annotated images not found on disk (first 5): {missing_images[:5]}")
    return samples


def load_coco_annotations(json_path, images_dir, image_index):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    id_to_name = {c["id"]: c["name"] for c in data.get("categories", [])}
    category_map = {cid: auto_map_category(cid, name) for cid, name in id_to_name.items()}
    print("COCO category mapping:")
    for cid, name in sorted(id_to_name.items()):
        mapped = category_map[cid]
        target = f"Class {mapped}: {CLASS_NAMES.get(mapped, '?')}" if mapped else "SKIP"
        print(f"  {cid}: {name} -> {target}")
    img_map = {img["id"]: img for img in data.get("images", [])}
    grouped = defaultdict(list)
    skipped = 0
    for ann in data.get("annotations", []):
        img_info = img_map.get(ann["image_id"])
        if not img_info:
            continue
        mapped = category_map.get(ann["category_id"])
        if mapped is None or mapped not in CLASS_NAMES:
            skipped += 1
            continue
        w, h = img_info.get("width", IMAGE_SIZE), img_info.get("height", IMAGE_SIZE)
        bbox = coco_bbox_to_pascal(ann["bbox"], w, h)
        if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
            continue
        fname = img_info["file_name"]
        grouped[fname].append({"bbox": bbox, "label": mapped})
    samples = []
    for fname, anns in grouped.items():
        img_path = resolve_image_path(fname, image_index, images_dir)
        if img_path:
            samples.append({"filename": fname, "path": img_path, "annotations": anns})
    print(f"COCO: loaded {len(samples)} images, skipped {skipped} annotations")
    return samples


# ── Load dataset ──
SEARCH_ROOTS = [TEST_IMAGES_DIR, WORK_DIR]
IMAGE_INDEX = build_image_index(SEARCH_ROOTS)
IMAGES_DIR = find_best_images_dir(SEARCH_ROOTS)

fmt = detect_format(ANN_PATH)
print(f"Detected annotation format: {fmt.upper()}")
print(f"Images directory: {IMAGES_DIR}")
print(f"Indexed {len(IMAGE_INDEX)} image name keys across {len(set(v[0] for v in IMAGE_INDEX.values()))} files")

if fmt == "csv":
    test_samples = load_csv_annotations(ANN_PATH, IMAGES_DIR, IMAGE_INDEX)
else:
    test_samples = load_coco_annotations(ANN_PATH, IMAGES_DIR, IMAGE_INDEX)

if not test_samples:
    print("\n❌ DEBUG INFO:")
    print(f"  ANN_PATH: {ANN_PATH}")
    print(f"  IMAGES_DIR: {IMAGES_DIR}")
    print(f"  Image folders: {scan_image_dirs(TEST_IMAGES_DIR)}")
    if fmt == "csv":
        df_dbg = pd.read_csv(ANN_PATH)
        df_dbg.columns = [c.strip().lower() for c in df_dbg.columns]
        for col in ("class", "class_id", "category", "label", "name"):
            if col in df_dbg.columns:
                print(f"  Unique '{col}' values: {df_dbg[col].dropna().unique()[:15].tolist()}")
                break
        else:
            print("  ⚠ No class column found — expected: class, class_id, category, label, or name")
    raise RuntimeError(
        "No test samples loaded. Common fixes:\n"
        "  1. Upload a zip whose images end up under test_images/test/\n"
        "  2. Ensure CSV 'filename' column matches image basenames\n"
        "  3. Check class column uses IDs 1-7, names like 'Caries', or Dataset_Batch2 IDs 0-8"
    )

class_counts = defaultdict(int)
for s in test_samples:
    for a in s["annotations"]:
        class_counts[a["label"]] += 1
print("\nTest set class distribution:")
for cid in CLASS_IDS:
    print(f"  {CLASS_NAMES[cid]}: {class_counts[cid]}")


## 5. Model Loading & Dataset


In [ ]:
def build_model(num_classes=NUM_CLASSES):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def load_checkpoint(model_path):
    try:
        ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    except TypeError:
        ckpt = torch.load(model_path, map_location=DEVICE)
    meta = {}
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
        meta = {k: ckpt[k] for k in ["train_losses", "val_losses", "best_val_loss", "class_names", "image_size", "test_results"] if k in ckpt}
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt
    return state_dict, meta


def get_val_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["class_labels"]))


class EvalDentalDataset(Dataset):
    def __init__(self, samples, transforms=None):
        self.samples = samples
        self.transforms = transforms

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = cv2.imread(sample["path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        bboxes = [a["bbox"] for a in sample["annotations"]]
        labels = [a["label"] for a in sample["annotations"]]
        if self.transforms:
            t = self.transforms(image=image, bboxes=bboxes, class_labels=labels)
            image = t["image"]
            bboxes = t["bboxes"]
            labels = t["class_labels"]
        target = {
            "boxes": torch.as_tensor(bboxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
            "orig_path": sample["path"],
            "filename": sample["filename"],
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


state_dict, checkpoint_meta = load_checkpoint(MODEL_PATH)
model = build_model()
model.load_state_dict(state_dict, strict=False)
model.to(DEVICE)
model.eval()

train_losses = checkpoint_meta.get("train_losses", [])
val_losses = checkpoint_meta.get("val_losses", [])

val_transform = get_val_transforms()
test_dataset = EvalDentalDataset(test_samples, transforms=val_transform)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"Loaded model from: {MODEL_PATH}")
print(f"Test images: {len(test_dataset)}")
if train_losses:
    print(f"Training history: {len(train_losses)} epochs in checkpoint")


## 6. Evaluation Engine


In [ ]:
def box_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0


def match_predictions(gt_boxes, gt_labels, pred_boxes, pred_labels, pred_scores, iou_thresh):
    matched_gt = set()
    tp, fp = [], []
    results = []
    order = sorted(range(len(pred_boxes)), key=lambda i: -pred_scores[i])
    for pi in order:
        pb, pl, ps = pred_boxes[pi], int(pred_labels[pi]), float(pred_scores[pi])
        best_iou, best_gi = 0.0, -1
        for gi, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
            if gi in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_gi = iou, gi
        if best_gi >= 0 and best_iou >= iou_thresh:
            gl = int(gt_labels[best_gi])
            matched_gt.add(best_gi)
            is_tp = pl == gl
            entry = {"type": "TP" if is_tp else "FP", "pred_label": pl, "gt_label": gl, "score": ps, "iou": best_iou, "pred_box": pb, "gt_box": gt_boxes[best_gi]}
            if is_tp:
                tp.append(entry)
            else:
                fp.append(entry)
            results.append(entry)
        else:
            entry = {"type": "FP", "pred_label": pl, "gt_label": None, "score": ps, "iou": best_iou, "pred_box": pb, "gt_box": None}
            fp.append(entry)
            results.append(entry)
    fn = []
    for gi, gl in enumerate(gt_labels):
        if gi not in matched_gt:
            fn.append({"type": "FN", "pred_label": None, "gt_label": int(gl), "score": 0.0, "iou": 0.0, "pred_box": None, "gt_box": gt_boxes[gi]})
    return tp, fp, fn, results


def compute_ap(recalls, precisions):
    recalls = np.concatenate([[0.0], recalls, [1.0]])
    precisions = np.concatenate([[0.0], precisions, [0.0]])
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])
    idx = np.where(recalls[1:] != recalls[:-1])[0]
    return float(np.sum((recalls[idx + 1] - recalls[idx]) * precisions[idx + 1]))


def compute_map_all(all_preds_by_class, num_gt_by_class, iou_thresh):
    aps = {}
    for cid in CLASS_IDS:
        preds = sorted(all_preds_by_class.get(cid, []), key=lambda x: -x["score"])
        n_gt = num_gt_by_class.get(cid, 0)
        if n_gt == 0:
            aps[cid] = float("nan")
            continue
        tp_cum, fp_cum = 0, 0
        precisions, recalls = [], []
        for p in preds:
            if p["matched"] and p["iou"] >= iou_thresh and p["pred_label"] == cid:
                tp_cum += 1
            else:
                fp_cum += 1
            precisions.append(tp_cum / (tp_cum + fp_cum) if (tp_cum + fp_cum) else 0)
            recalls.append(tp_cum / n_gt)
        aps[cid] = compute_ap(np.array(recalls), np.array(precisions)) if preds else 0.0
    valid = [v for v in aps.values() if not np.isnan(v)]
    return float(np.mean(valid)) if valid else 0.0, aps


def save_fig(name, dpi=150):
    path = os.path.join(METRICS_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    return path


@torch.no_grad()
def run_full_evaluation(model, loader, conf=CONF_THRESHOLD, iou_thresh=IOU_THRESHOLD):
    model.eval()
    per_class_tp = defaultdict(int)
    per_class_fp = defaultdict(int)
    per_class_fn = defaultdict(int)
    per_class_support = defaultdict(int)
    per_class_iou_sum = defaultdict(float)
    per_class_iou_count = defaultdict(int)
    all_preds_by_class = defaultdict(list)
    num_gt_by_class = defaultdict(int)
    y_true_inst, y_pred_inst = [], []
    cm_pairs = []
    all_confidences = []
    all_tp_ious = []
    inference_times = []
    error_examples = {"FP": [], "FN": [], "low_conf_tp": []}
    image_records = []

    for images, targets in tqdm(loader, desc="Evaluating"):
        t0 = time.perf_counter()
        outputs = model([img.to(DEVICE) for img in images])
        batch_time = (time.perf_counter() - t0) / len(images)
        for output, target in zip(outputs, targets):
            inference_times.append(batch_time)
            gt_boxes = target["boxes"].cpu().numpy()
            gt_labels = target["labels"].cpu().numpy()
            for gl in gt_labels:
                per_class_support[int(gl)] += 1
                num_gt_by_class[int(gl)] += 1
            mask = output["scores"].cpu().numpy() >= conf
            pred_boxes = output["boxes"].cpu().numpy()[mask]
            pred_labels = output["labels"].cpu().numpy()[mask]
            pred_scores = output["scores"].cpu().numpy()[mask]
            all_confidences.extend(pred_scores.tolist())
            tp, fp, fn, _ = match_predictions(gt_boxes, gt_labels, pred_boxes, pred_labels, pred_scores, iou_thresh)
            for e in tp:
                per_class_tp[e["pred_label"]] += 1
                per_class_iou_sum[e["pred_label"]] += e["iou"]
                per_class_iou_count[e["pred_label"]] += 1
                all_tp_ious.append(e["iou"])
                y_true_inst.append(e["gt_label"])
                y_pred_inst.append(e["pred_label"])
                cm_pairs.append((e["gt_label"], e["pred_label"]))
                error_examples["low_conf_tp"].append({**e, "path": target["orig_path"], "filename": target["filename"]})
            for e in fp:
                per_class_fp[e["pred_label"]] += 1
                error_examples["FP"].append({**e, "path": target["orig_path"], "filename": target["filename"]})
            for e in fn:
                per_class_fn[e["gt_label"]] += 1
                y_true_inst.append(e["gt_label"])
                y_pred_inst.append(-1)
                error_examples["FN"].append({**e, "path": target["orig_path"], "filename": target["filename"]})
            for pi, (pb, pl, ps) in enumerate(zip(pred_boxes, pred_labels, pred_scores)):
                best_iou, matched = 0.0, False
                for gb, gl in zip(gt_boxes, gt_labels):
                    iou = box_iou(pb, gb)
                    if iou > best_iou:
                        best_iou = iou
                        matched = int(pl) == int(gl) and iou >= iou_thresh
                all_preds_by_class[int(pl)].append({"score": float(ps), "matched": matched, "iou": best_iou, "pred_label": int(pl)})
            image_records.append({"path": target["orig_path"], "filename": target["filename"], "gt_boxes": gt_boxes, "gt_labels": gt_labels, "pred_boxes": pred_boxes, "pred_labels": pred_labels, "pred_scores": pred_scores})

    map50, ap50_per_class = compute_map_all(all_preds_by_class, num_gt_by_class, 0.5)
    map5095_list, ap5095_per_class = [], {c: [] for c in CLASS_IDS}
    for t in IOU_THRESHOLDS_MAP:
        m, aps = compute_map_all(all_preds_by_class, num_gt_by_class, t)
        map5095_list.append(m)
        for c in CLASS_IDS:
            ap5095_per_class[c].append(aps.get(c, 0.0))
    map5095 = float(np.mean(map5095_list))

    per_class = {}
    for cid in CLASS_IDS:
        tp = per_class_tp[cid]
        fp = per_class_fp[cid]
        fn = per_class_fn[cid]
        support = per_class_support[cid]
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        total_gt = sum(per_class_support.values())
        tn = max(0, total_gt - support)
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        fnr = fn / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        avg_iou = per_class_iou_sum[cid] / per_class_iou_count[cid] if per_class_iou_count[cid] else 0.0
        per_class[cid] = {"precision": prec, "recall": rec, "f1": f1, "specificity": spec, "fpr": fpr, "fnr": fnr, "support": support, "tp": tp, "fp": fp, "fn": fn, "ap50": ap50_per_class.get(cid, 0.0), "avg_iou": avg_iou, "ap5095": float(np.nanmean(ap5095_per_class[cid])) if ap5095_per_class[cid] else 0.0}

    y_true_arr = np.array(y_true_inst)
    y_pred_arr = np.array(y_pred_inst)
    valid_mask = y_pred_arr >= 0
    y_t = y_true_arr[valid_mask]
    y_p = y_pred_arr[valid_mask]
    labels_idx = list(range(len(CLASS_IDS)))
    label_ids = CLASS_IDS

    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_t, y_p, labels=label_ids, average="macro", zero_division=0)
    prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(y_t, y_p, labels=label_ids, average="micro", zero_division=0)
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(y_t, y_p, labels=label_ids, average="weighted", zero_division=0)
    accuracy = accuracy_score(y_t, y_p) if len(y_t) else 0.0
    balanced_acc = balanced_accuracy_score(y_t, y_p) if len(y_t) else 0.0
    mcc = matthews_corrcoef(y_t, y_p) if len(y_t) else 0.0
    cm = confusion_matrix(y_t, y_p, labels=label_ids)

    roc_auc_per_class = {}
    pr_curves = {}
    roc_curves = {}
    for cid in CLASS_IDS:
        y_bin = (y_true_arr == cid).astype(int)
        scores = []
        for rec in image_records:
            best = 0.0
            for pb, pl, ps in zip(rec["pred_boxes"], rec["pred_labels"], rec["pred_scores"]):
                if int(pl) == cid:
                    best = max(best, float(ps))
            scores.append(best)
        scores = np.array(scores)
        if len(np.unique(y_bin)) > 1:
            try:
                roc_auc_per_class[cid] = roc_auc_score(y_bin, scores)
                fpr_c, tpr_c, _ = roc_curve(y_bin, scores)
                roc_curves[cid] = (fpr_c, tpr_c)
            except ValueError:
                roc_auc_per_class[cid] = float("nan")
        else:
            roc_auc_per_class[cid] = float("nan")
        gt_scores = []
        gt_labels_bin = []
        for rec in image_records:
            has_class = any(int(gl) == cid for gl in rec["gt_labels"])
            best = 0.0
            for pb, pl, ps in zip(rec["pred_boxes"], rec["pred_labels"], rec["pred_scores"]):
                if int(pl) == cid:
                    best = max(best, float(ps))
            gt_labels_bin.append(1 if has_class else 0)
            gt_scores.append(best)
        if sum(gt_labels_bin) > 0:
            prec_c, rec_c, _ = precision_recall_curve(gt_labels_bin, gt_scores)
            pr_curves[cid] = (rec_c, prec_c)

    error_examples["FP"] = sorted(error_examples["FP"], key=lambda x: -x["score"])[:5]
    error_examples["FN"] = sorted(error_examples["FN"], key=lambda x: -x.get("score", 0))[:5]
    error_examples["low_conf_tp"] = sorted(error_examples["low_conf_tp"], key=lambda x: x["score"])[:5]

    return {
        "per_class": per_class,
        "overall": {
            "accuracy": accuracy, "precision_macro": prec_macro, "precision_micro": prec_micro, "precision_weighted": prec_weighted,
            "recall_macro": rec_macro, "recall_micro": rec_micro, "recall_weighted": rec_weighted,
            "f1_macro": f1_macro, "f1_micro": f1_micro, "f1_weighted": f1_weighted,
            "mcc": mcc, "balanced_accuracy": balanced_acc,
            "map50": map50, "map5095": map5095,
        },
        "roc_auc_per_class": roc_auc_per_class,
        "confusion_matrix": cm,
        "pr_curves": pr_curves,
        "roc_curves": roc_curves,
        "all_confidences": all_confidences,
        "all_tp_ious": all_tp_ious,
        "inference_times": inference_times,
        "error_examples": error_examples,
        "image_records": image_records,
        "class_counts": dict(class_counts),
    }


print("Running full evaluation...")
results = run_full_evaluation(model, test_loader)
print("Evaluation complete.")


## 7. Visualizations


In [ ]:
saved_figs = []

# 1. Loss curves
if train_losses and val_losses:
    fig, ax = plt.subplots(figsize=(10, 5))
    epochs = range(1, len(train_losses) + 1)
    ax.plot(epochs, train_losses, "b-o", label="Train Loss", markersize=4)
    ax.plot(epochs, val_losses, "r-s", label="Val Loss", markersize=4)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training vs Validation Loss")
    ax.legend()
    saved_figs.append(save_fig("01_loss_curves.png"))
else:
    print("No training history in checkpoint — skipping loss curves.")

# 2. Precision-Recall curves per class
fig, ax = plt.subplots(figsize=(10, 7))
for cid in CLASS_IDS:
    if cid in results["pr_curves"]:
        rec_c, prec_c = results["pr_curves"][cid]
        ax.plot(rec_c, prec_c, label=CLASS_NAMES[cid])
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves (Per Class)")
ax.legend(fontsize=8, loc="lower left")
saved_figs.append(save_fig("02_precision_recall_curves.png"))

# 3. ROC curves per class
fig, ax = plt.subplots(figsize=(10, 7))
for cid in CLASS_IDS:
    if cid in results["roc_curves"]:
        fpr_c, tpr_c = results["roc_curves"][cid]
        auc_val = results["roc_auc_per_class"].get(cid, float("nan"))
        ax.plot(fpr_c, tpr_c, label=f"{CLASS_NAMES[cid]} (AUC={auc_val:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves (One-vs-Rest)")
ax.legend(fontsize=8, loc="lower right")
saved_figs.append(save_fig("03_roc_curves.png"))

# 4. Confusion matrix heatmaps
cm = results["confusion_matrix"]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0], cmap="Blues")
axes[0].set_title("Confusion Matrix (Counts)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
sns.heatmap(cm_norm, annot=True, fmt=".1%", xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[1], cmap="Blues")
axes[1].set_title("Confusion Matrix (Normalized %)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
saved_figs.append(save_fig("04_confusion_matrix.png"))

# Per-class confusion matrices
for i, cid in enumerate(CLASS_IDS):
    row = cm[i] if i < len(cm) else np.zeros(len(CLASS_IDS))
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.heatmap(row.reshape(1, -1), annot=True, fmt="d", xticklabels=CLASS_LABELS, yticklabels=[CLASS_NAMES[cid]], ax=ax, cmap="Oranges")
    ax.set_title(f"Confusion Row: {CLASS_NAMES[cid]}")
    saved_figs.append(save_fig(f"04b_confusion_row_class{cid}.png"))

# 5. Per-class performance bar chart
metrics_df = pd.DataFrame(results["per_class"]).T
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(CLASS_IDS))
w = 0.25
ax.bar(x - w, [results["per_class"][c]["precision"] for c in CLASS_IDS], w, label="Precision")
ax.bar(x, [results["per_class"][c]["recall"] for c in CLASS_IDS], w, label="Recall")
ax.bar(x + w, [results["per_class"][c]["f1"] for c in CLASS_IDS], w, label="F1")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_LABELS, rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Per-Class Performance")
ax.legend()
saved_figs.append(save_fig("05_per_class_performance.png"))

# 6. Confidence distribution
if results["all_confidences"]:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(results["all_confidences"], bins=30, color="steelblue", edgecolor="white")
    ax.axvline(CONF_THRESHOLD, color="red", linestyle="--", label=f"Threshold={CONF_THRESHOLD}")
    ax.set_xlabel("Confidence Score")
    ax.set_ylabel("Count")
    ax.set_title("Detection Confidence Distribution")
    ax.legend()
    saved_figs.append(save_fig("06_confidence_distribution.png"))

# 7. IoU distribution for correct detections
if results["all_tp_ious"]:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(results["all_tp_ious"], bins=20, color="seagreen", edgecolor="white")
    ax.axvline(IOU_THRESHOLD, color="red", linestyle="--", label=f"IoU threshold={IOU_THRESHOLD}")
    ax.set_xlabel("IoU")
    ax.set_ylabel("Count")
    ax.set_title("IoU Distribution (True Positives)")
    ax.legend()
    saved_figs.append(save_fig("07_iou_distribution.png"))

# 8. FP/FN analysis
fig, ax = plt.subplots(figsize=(12, 6))
fp_vals = [results["per_class"][c]["fp"] for c in CLASS_IDS]
fn_vals = [results["per_class"][c]["fn"] for c in CLASS_IDS]
x = np.arange(len(CLASS_IDS))
ax.bar(x - 0.2, fp_vals, 0.4, label="False Positives", color="tomato")
ax.bar(x + 0.2, fn_vals, 0.4, label="False Negatives", color="orange")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_LABELS, rotation=30, ha="right")
ax.set_title("False Positives & False Negatives per Class")
ax.legend()
saved_figs.append(save_fig("08_fp_fn_analysis.png"))

# 9. Learning curves (same as loss if available)
if train_losses:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(1, len(train_losses)+1), train_losses, label="Train")
    if val_losses:
        ax.plot(range(1, len(val_losses)+1), val_losses, label="Val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Learning Curves")
    ax.legend()
    saved_figs.append(save_fig("09_learning_curves.png"))

# 10. Class distribution
fig, ax = plt.subplots(figsize=(10, 5))
counts = [results["class_counts"].get(c, 0) for c in CLASS_IDS]
ax.bar(CLASS_LABELS, counts, color=sns.color_palette("husl", len(CLASS_IDS)))
ax.set_title("Class Distribution in Test Set")
ax.set_ylabel("Instance Count")
plt.xticks(rotation=30, ha="right")
saved_figs.append(save_fig("10_class_distribution.png"))

print(f"Saved {len(saved_figs)} figures to {METRICS_DIR}")


## 8. Error Analysis & Inference Timing


In [ ]:
def draw_boxes(path, gt_boxes, gt_labels, pred_boxes, pred_labels, pred_scores, title=""):
    img = Image.open(path).convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
    draw = ImageDraw.Draw(img)
    for box, label in zip(gt_boxes, gt_labels):
        draw.rectangle(box, outline="lime", width=2)
        draw.text((box[0], box[1]), f"GT:{CLASS_NAMES.get(int(label), label)}", fill="lime")
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        draw.rectangle(box, outline="red", width=2)
        draw.text((box[0], max(0, box[1]-12)), f"P:{CLASS_NAMES.get(int(label), label)} {score:.2f}", fill="red")
    out_path = os.path.join(METRICS_DIR, os.path.basename(path).rsplit(".", 1)[0] + f"_{title}.png")
    img.save(out_path)
    return out_path


# Error analysis visualizations
for i, ex in enumerate(results["error_examples"]["FP"]):
    rec = next(r for r in results["image_records"] if r["path"] == ex["path"])
    draw_boxes(ex["path"], rec["gt_boxes"], rec["gt_labels"], [ex["pred_box"]], [ex["pred_label"]], [ex["score"]], f"FP_{i+1}")

for i, ex in enumerate(results["error_examples"]["FN"]):
    rec = next(r for r in results["image_records"] if r["path"] == ex["path"])
    draw_boxes(ex["path"], [ex["gt_box"]], [ex["gt_label"]], [], [], [], f"FN_{i+1}")

for i, ex in enumerate(results["error_examples"]["low_conf_tp"]):
    draw_boxes(ex["path"], [ex["gt_box"]], [ex["gt_label"]], [ex["pred_box"]], [ex["pred_label"]], [ex["score"]], f"lowTP_{i+1}")

# Inference timing
times = results["inference_times"]
avg_time = float(np.mean(times)) if times else 0.0
fps = 1.0 / avg_time if avg_time > 0 else 0.0
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(times, bins=20, color="purple", edgecolor="white", alpha=0.8)
ax.axvline(avg_time, color="black", linestyle="--", label=f"Mean={avg_time*1000:.1f} ms")
ax.set_xlabel("Inference Time (seconds)")
ax.set_ylabel("Count")
ax.set_title(f"Inference Time Distribution (FPS={fps:.2f})")
ax.legend()
save_fig("11_inference_time.png")

print(f"Average inference time: {avg_time*1000:.2f} ms/image")
print(f"FPS: {fps:.2f}")


## 9. Model Summary


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)

print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(f"Architecture:        Faster R-CNN ResNet50-FPN")
print(f"Total parameters:    {total_params:,}")
print(f"Trainable parameters:{trainable_params:,}")
print(f"Model file size:     {model_size_mb:.2f} MB")
print(f"Num classes:         {NUM_CLASSES} (incl. background)")
print(f"Input size:          {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Checkpoint:          {os.path.basename(MODEL_PATH)}")
if checkpoint_meta.get("best_val_loss") is not None:
    print(f"Best val loss:       {checkpoint_meta['best_val_loss']:.4f}")

# Architecture summary
print("\n--- Layer Summary (first 20 modules) ---")
for i, (name, module) in enumerate(model.named_children()):
    if i >= 5:
        print("  ...")
        break
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {module.__class__.__name__} ({n_params:,} params)")


## 10. Comprehensive Summary Table & HTML Report


In [ ]:
ov = results["overall"]
pc = results["per_class"]

# Build per-class DataFrame
rows = []
for cid in CLASS_IDS:
    m = pc[cid]
    rows.append({
        "Class ID": cid,
        "Class Name": CLASS_NAMES[cid],
        "Support": m["support"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1 Score": m["f1"],
        "Specificity": m["specificity"],
        "FPR": m["fpr"],
        "FNR": m["fnr"],
        "AP@0.50": m["ap50"],
        "AP@0.50:0.95": m["ap5095"],
        "Avg IoU (TP)": m["avg_iou"],
        "ROC-AUC": results["roc_auc_per_class"].get(cid, float("nan")),
        "TP": m["tp"], "FP": m["fp"], "FN": m["fn"],
    })
summary_df = pd.DataFrame(rows)

print("=" * 100)
print("COMPREHENSIVE PERFORMANCE SUMMARY")
print("=" * 100)
print("\n--- OVERALL METRICS ---")
overall_table = pd.DataFrame([
    {"Metric": "Accuracy", "Value": f"{ov['accuracy']:.4f}"},
    {"Metric": "Balanced Accuracy", "Value": f"{ov['balanced_accuracy']:.4f}"},
    {"Metric": "Matthews Correlation Coefficient", "Value": f"{ov['mcc']:.4f}"},
    {"Metric": "Precision (macro)", "Value": f"{ov['precision_macro']:.4f}"},
    {"Metric": "Precision (micro)", "Value": f"{ov['precision_micro']:.4f}"},
    {"Metric": "Precision (weighted)", "Value": f"{ov['precision_weighted']:.4f}"},
    {"Metric": "Recall (macro)", "Value": f"{ov['recall_macro']:.4f}"},
    {"Metric": "Recall (micro)", "Value": f"{ov['recall_micro']:.4f}"},
    {"Metric": "Recall (weighted)", "Value": f"{ov['recall_weighted']:.4f}"},
    {"Metric": "F1 (macro)", "Value": f"{ov['f1_macro']:.4f}"},
    {"Metric": "F1 (micro)", "Value": f"{ov['f1_micro']:.4f}"},
    {"Metric": "F1 (weighted)", "Value": f"{ov['f1_weighted']:.4f}"},
    {"Metric": "mAP @ IoU=0.50", "Value": f"{ov['map50']:.4f}"},
    {"Metric": "mAP @ IoU=0.50:0.95", "Value": f"{ov['map5095']:.4f}"},
    {"Metric": "Avg Inference Time (ms)", "Value": f"{avg_time*1000:.2f}"},
    {"Metric": "FPS", "Value": f"{fps:.2f}"},
    {"Metric": "Total Parameters", "Value": f"{total_params:,}"},
    {"Metric": "Model Size (MB)", "Value": f"{model_size_mb:.2f}"},
])
print(overall_table.to_string(index=False))

print("\n--- PER-CLASS METRICS ---")
display_cols = ["Class Name", "Support", "Precision", "Recall", "F1 Score", "Specificity", "FPR", "FNR", "AP@0.50", "Avg IoU (TP)", "ROC-AUC", "TP", "FP", "FN"]
print(summary_df[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

summary_df.to_csv(os.path.join(METRICS_DIR, "per_class_metrics.csv"), index=False)
overall_table.to_csv(os.path.join(METRICS_DIR, "overall_metrics.csv"), index=False)

# HTML report
html_parts = [
    "<html><head><meta charset='utf-8'><title>DENTRAT Performance Report</title>",
    "<style>body{font-family:Arial,sans-serif;margin:40px;background:#f8f9fa;}",
    "h1,h2{color:#2c3e50;} table{border-collapse:collapse;width:100%;margin:16px 0;}",
    "th,td{border:1px solid #ddd;padding:8px;text-align:left;} th{background:#3498db;color:white;}",
    "tr:nth-child(even){background:#f2f2f2;} .metric-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;}",
    ".card{background:white;padding:16px;border-radius:8px;box-shadow:0 2px 4px rgba(0,0,0,0.1);}",
    "img{max-width:100%;margin:12px 0;border:1px solid #ddd;border-radius:4px;}</style></head><body>",
    f"<h1>DENTRAT Performance Report</h1>",
    f"<p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>",
    f"<p>Model: <code>{os.path.basename(MODEL_PATH)}</code> | Test images: {len(test_samples)} | Device: {DEVICE}</p>",
    "<h2>Overall Metrics</h2><div class='metric-grid'>",
]
for _, row in overall_table.iterrows():
    html_parts.append(f"<div class='card'><strong>{row['Metric']}</strong><br>{row['Value']}</div>")
html_parts.append("</div><h2>Per-Class Metrics</h2>")
html_parts.append(summary_df[display_cols].to_html(index=False, float_format=lambda x: f"{x:.4f}"))
html_parts.append("<h2>Visualizations</h2>")
for fig_name in sorted(os.listdir(METRICS_DIR)):
    if fig_name.endswith(".png"):
        html_parts.append(f"<h3>{fig_name}</h3><img src='{fig_name}' alt='{fig_name}'>")
html_parts.append("</body></html>")

report_path = os.path.join(METRICS_DIR, "performance_report.html")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(html_parts))

# Markdown summary
try:
    md_report = ["# DENTRAT Performance Report\n", f"Generated: {datetime.now()}\n\n", "## Overall Metrics\n", overall_table.to_markdown(index=False), "\n## Per-Class Metrics\n", summary_df[display_cols].to_markdown(index=False, floatfmt=".4f")]
except ImportError:
    md_report = ["# DENTRAT Performance Report\n", f"Generated: {datetime.now()}\n\n", summary_df[display_cols].to_csv(index=False)]
with open(os.path.join(METRICS_DIR, "performance_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(md_report))

print(f"\nReport saved: {report_path}")
print(f"CSV metrics saved to {METRICS_DIR}")


## 11. Download Results (Colab)


In [ ]:
if IN_COLAB:
    import shutil
    zip_path = os.path.join(WORK_DIR, "metrics_output.zip")
    shutil.make_archive(os.path.join(WORK_DIR, "metrics_output"), "zip", METRICS_DIR)
    print(f"Created {zip_path}")
    files.download(zip_path)
    print("Download started — check your browser downloads.")
else:
    print(f"Results available in: {METRICS_DIR}")
